In [1]:
# Extract images for all items in the items in the dataset

import json
metadata_dict = {}
file ="data/meta_All_Beauty.jsonl" # e.g., "All_Beauty.jsonl", downloaded from the `review` link above
with open(file, 'r') as fp:
    for line in fp:
        # try:
        tmp_dict = {'images':[]}
        data = json.loads(line.strip())
        ctr =0
        for img_dict in data['images']:
            # if type(tmp_dict['images']) == list:
            tmp_dict['images'].append(img_dict.get("large", img_dict.get("hi-res", "")))
            
        # except:
        #     print(line.strip())
        metadata_dict[data['parent_asin']] = tmp_dict

In [14]:
metadata_dict['B09FNTZFZZ']

{'images': ['https://m.media-amazon.com/images/I/31sfavSwQ0L.jpg',
  'https://m.media-amazon.com/images/I/21ic+KeB2AL.jpg',
  'https://m.media-amazon.com/images/I/41cimgd06KL.jpg',
  'https://m.media-amazon.com/images/I/51ergpeDhfL.jpg',
  'https://m.media-amazon.com/images/I/41jWGfznN4L.jpg']}

In [ ]:
import requests, os
from concurrent.futures import ThreadPoolExecutor

folder_location = "data/All_Beauty/"

os.makedirs(folder_location, exist_ok=True)

def download_image(product_id, idx, image_url, folder_location):
    try:
        # Fetch the image
        response = requests.get(image_url, stream=True)
        response.raise_for_status()  # Raise an error for bad status codes

        # Create a unique filename
        filename = f"{product_id}_{idx + 1}.jpg"
        file_path = os.path.join(folder_location, filename)

        # Save the image
        with open(file_path, 'wb') as file:
            for chunk in response.iter_content(1024):
                file.write(chunk)
        print(f"Downloaded: {file_path}")
    except Exception as e:
        print(f"Failed to download {image_url}: {e}")

# Use ThreadPoolExecutor for parallel downloads
with ThreadPoolExecutor(max_workers=32) as executor:
    futures = []
    folder_content = os.listdir(folder_location)
    # Check if images already exist
    existing_images = [f.split("_")[0] for f in os.listdir(folder_location) if f.endswith('.jpg')]
    existing_images = list(set(existing_images))
    
    
    with open("data/s.csv", "r") as f:
        file_contents = f.read()
    contents_list = file_contents.split(",")
    
    
    for product_id in contents_list:
        # Download up to 2 images
        for idx, image_url in enumerate(metadata_dict[product_id]['images'][:2]):
            futures.append(executor.submit(download_image, product_id, idx, image_url, folder_location))